# Download Dataset

In [ ]:
# Import library
import os
import kagglehub
import pandas as pd
import re

pd.set_option('display.width', 200)

In [ ]:
# Import library
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import joblib

In [ ]:
path = kagglehub.dataset_download("sharmajicoder/student-mental-health-and-burnout")
path += "/student_mental_health_burnout_1M.csv"

100%|██████████| 131M/131M [00:01<00:00, 70.6MB/s]

Extracting files...


In [ ]:
file = pd.read_csv(path)

In [ ]:
# Sample 2% (20k) of the data randomly
train_df = file.sample(frac=0.02, random_state=42)

In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20000 entries, 987231 to 809896
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age                   20000 non-null  int64  
 1   gender                20000 non-null  object 
 2   academic_year         20000 non-null  int64  
 3   study_hours_per_day   20000 non-null  float64
 4   exam_pressure         20000 non-null  float64
 5   academic_performance  20000 non-null  float64
 6   stress_level          20000 non-null  float64
 7   anxiety_score         20000 non-null  float64
 8   depression_score      20000 non-null  float64
 9   sleep_hours           20000 non-null  float64
 10  physical_activity     20000 non-null  float64
 11  social_support        20000 non-null  float64
 12  screen_time           20000 non-null  float64
 13  internet_usage        20000 non-null  float64
 14  financial_stress      20000 non-null  float64
 15  family_expectation

In [ ]:
print('Missing values per column:')
print(train_df.isnull().sum())

Missing values per column:
age                     0
gender                  0
academic_year           0
study_hours_per_day     0
exam_pressure           0
academic_performance    0
stress_level            0
anxiety_score           0
depression_score        0
sleep_hours             0
physical_activity       0
social_support          0
screen_time             0
internet_usage          0
financial_stress        0
family_expectation      0
burnout_score           0
mental_health_index     0
risk_level              0
dropout_risk            0
dtype: int64


In [ ]:
# Sample 50% (10k) of the data randomly
rf_df = file.sample(frac=0.5, random_state=42)

## Outlier

In [ ]:
# Check the min and max value for outlier
num_cols = rf_df.select_dtypes(include='number').columns
rf_df[num_cols].agg(['min', 'max']).T

,min,max
age,17.000000,29.000000
academic_year,1.000000,4.000000
study_hours_per_day,0.000000,14.000000
exam_pressure,1.000000,10.000000
academic_performance,42.365714,97.246309
stress_level,0.000000,10.000000
anxiety_score,0.000000,9.698745
depression_score,0.000000,8.018870
sleep_hours,3.000000,10.000000
physical_activity,0.000000,7.000000


In [ ]:
print('Missing values per column:')
print(rf_df.isnull().sum())

Missing values per column:
age                     0
gender                  0
academic_year           0
study_hours_per_day     0
exam_pressure           0
academic_performance    0
stress_level            0
anxiety_score           0
depression_score        0
sleep_hours             0
physical_activity       0
social_support          0
screen_time             0
internet_usage          0
financial_stress        0
family_expectation      0
burnout_score           0
mental_health_index     0
risk_level              0
dropout_risk            0
dtype: int64


## Simulasi Missing value

In [ ]:
rng = np.random.default_rng(42)

demo_df = rf_df.drop(columns='anxiety_score').copy()
text_cols = ['study_hours_per_day', 'sleep_hours', 'screen_time', 'internet_usage']
formats = ['{}', '{} hour', '{} hours', '{}hrs', '{} H']
paragraphs = ['Depends on the day, sometimes a lot sometimes barely any',
              'Not sure, I never really counted',
              'Quite a lot during exam season but less otherwise']

In [ ]:
for col in text_cols:
    fmt = rng.choice(formats, size=len(demo_df))
    demo_df[col] = [f.format(f'{v:g}') for f, v in zip(fmt, demo_df[col].round(1))]
    para = rng.random(len(demo_df)) < 0.01  # 1% paragraph answers
    demo_df.loc[para, col] = rng.choice(paragraphs, size=para.sum())

In [ ]:
for col in demo_df.columns:
    rate = rng.uniform(0.02, 0.10)
    demo_df.loc[rng.random(len(demo_df)) < rate, col] = np.nan

print(demo_df[text_cols].sample(10, random_state=1))

       study_hours_per_day sleep_hours screen_time internet_usage
637585            5.6 hour         6.3   3.5 hours            4.1
958924           2.8 hours         NaN       3.3 H          3.1 H
487953           5.5 hours      7.9hrs   5.8 hours         7.2hrs
369933            6.9 hour    7.4 hour           4            5 H
454224               6.5 H       4.9 H     7 hours            7 H
125534               6.5 H   8.7 hours      6 hour            4.9
411565               6.2 H    4.5 hour    2.1 hour            2.4
427062            3.7 hour       8.1 H         NaN            7.4
519823                 3 H       5.2 H       1.2 H            1.7
84869               7.2hrs       5.3 H         4.7       5.4 hour


In [ ]:
demo_df.head(10)

,age,gender,academic_year,study_hours_per_day,exam_pressure,academic_performance,stress_level,depression_score,sleep_hours,physical_activity,social_support,screen_time,internet_usage,financial_stress,family_expectation,burnout_score,mental_health_index,risk_level,dropout_risk
987231,19.0,Male,1.0,3.2,4.638069,70.502920,3.341312,0.719136,6,NaN,4.999102,5.8,NaN,NaN,6.048954,0.615961,8.417823,NaN,0.385568
79954,21.0,NaN,1.0,7.9hrs,6.760116,73.247098,5.347451,0.024451,6.2 H,2.170758,8.157403,7.2hrs,6hrs,1.829564,7.372226,2.618701,6.528244,Low,NaN
567130,17.0,Female,4.0,3.2hrs,5.691440,69.259244,3.566456,0.000000,7.1hrs,NaN,7.306309,2.5 hours,3.5,6.945419,5.545514,0.000000,7.829562,NaN,NaN
500891,17.0,Male,3.0,5.6 hours,NaN,68.375104,NaN,1.168679,3.1hrs,NaN,NaN,3hrs,3.1,7.278448,2.576577,4.608672,5.928762,Medium,1.442608
55399,NaN,Male,4.0,2.6 hours,4.030860,67.057743,3.072377,0.549456,6.2 hour,2.859210,NaN,7.7 hours,7.8,6.473021,4.199195,0.057824,8.178880,Low,0.000000
135049,18.0,Male,1.0,0.6 H,4.864401,64.995896,6.256976,1.871557,3.6 H,2.348501,6.108822,"Not sure, I never really counted",NaN,6.778194,8.762637,3.733832,5.543427,Medium,2.157004
733378,26.0,Other,3.0,7.4,8.067873,70.205509,5.746703,2.775329,8.9,4.613276,5.125024,4.9 H,3.4 hour,8.372653,7.607321,1.874080,NaN,Low,3.464320
732057,28.0,Female,2.0,5.3hrs,5.858209,77.140076,0.857875,0.335719,8.9 hours,1.466787,NaN,4.2 hour,4.3 hour,3.386778,4.907440,0.000000,NaN,NaN,0.878914
51333,20.0,Female,1.0,8 hour,8.748923,70.862742,4.033817,0.000000,6.2,3.600459,8.082812,NaN,5.8 H,2.419594,7.608282,1.174205,7.618454,Low,NaN
731479,18.0,Male,4.0,5.9,5.504990,79.849741,3.519604,1.027579,5.5 hours,2.998635,4.837851,5 hours,4.7hrs,7.002925,4.955608,1.076145,7.714114,Low,2.083791


## Cleaning

In [ ]:
pattern = r'^\s*(\d+(?:\.\d+)?)\s*(?:hours?|hrs?|h)?\s*$'
clean_demo = demo_df.copy()

for col in text_cols:
    extracted = clean_demo[col].str.extract(pattern, flags=re.IGNORECASE)[0]
    clean_demo[col] = pd.to_numeric(extracted)

print("messy data:\n", demo_df[text_cols].isna().sum())
print("\ncleaned data:\n", clean_demo[text_cols].isna().sum())
print("\nunreadable (dropped):\n", clean_demo[text_cols].isna().sum() - demo_df[text_cols].isna().sum())

messy data:
 study_hours_per_day    23923
sleep_hours            22510
screen_time            12913
internet_usage         13831
dtype: int64

cleaned data:
 study_hours_per_day    28607
sleep_hours            27108
screen_time            17782
internet_usage         18838
dtype: int64

unreadable (dropped):
 study_hours_per_day    4684
sleep_hours            4598
screen_time            4869
internet_usage         5007
dtype: int64


In [ ]:
num = clean_demo.select_dtypes('number').columns
cat = clean_demo.select_dtypes(exclude='number').columns

clean_demo[num] = clean_demo[num].fillna(clean_demo[num].median())
clean_demo[cat] = clean_demo[cat].fillna(clean_demo[cat].mode().iloc[0])

print(clean_demo.isna().sum())

age                     0
gender                  0
academic_year           0
study_hours_per_day     0
exam_pressure           0
academic_performance    0
stress_level            0
depression_score        0
sleep_hours             0
physical_activity       0
social_support          0
screen_time             0
internet_usage          0
financial_stress        0
family_expectation      0
burnout_score           0
mental_health_index     0
risk_level              0
dropout_risk            0
dtype: int64


In [ ]:
# Create a copy to avoid modifying the original 'file' DataFrame for ML
df_ml = clean_demo.copy()

# Define features (X)
features = [
    'age', 'gender', 'study_hours_per_day', 'academic_performance', 'exam_pressure',
    'stress_level', 'sleep_hours', 'physical_activity',
    'financial_stress', 'family_expectation', 'burnout_score'
]

X = df_ml[features]

In [ ]:
# Transformasi (Encoding)
le_gender = LabelEncoder()
X.loc[:, 'gender'] = le_gender.fit_transform(X['gender'])

# check mapping
dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_)))

{'Female': np.int64(0), 'Male': np.int64(1), 'Other': np.int64(2)}

In [ ]:
X.head()

,age,gender,study_hours_per_day,academic_performance,exam_pressure,stress_level,sleep_hours,physical_activity,financial_stress,family_expectation,burnout_score
987231,19.0,1,3.2,70.502920,4.638069,3.341312,6.0,2.995741,4.998297,6.048954,0.615961
79954,21.0,0,7.9,73.247098,6.760116,5.347451,6.2,2.170758,1.829564,7.372226,2.618701
567130,17.0,0,3.2,69.259244,5.691440,3.566456,7.1,2.995741,6.945419,5.545514,0.000000
500891,17.0,1,5.6,68.375104,6.001107,4.246895,3.1,2.995741,7.278448,2.576577,4.608672
55399,23.0,1,2.6,67.057743,4.030860,3.072377,6.2,2.859210,6.473021,4.199195,0.057824


In [ ]:
def engineer(df):
    df = df.copy()
    df['free_hours'] = 24 - df['study_hours_per_day'] - df['sleep_hours']
    df['study_sleep_ratio'] = df['study_hours_per_day'] / df['sleep_hours'].clip(lower=1)
    df['external_pressure'] = df[['exam_pressure', 'financial_stress', 'family_expectation']].mean(axis=1)
    return df

In [ ]:
demo_X = engineer(X)

demo_X[['free_hours', 'study_sleep_ratio', 'external_pressure']].head()

,free_hours,study_sleep_ratio,external_pressure
987231,14.8,0.533333,5.228440
79954,9.9,1.274194,5.320635
567130,13.7,0.450704,6.060791
500891,15.3,1.806452,5.285377
55399,15.2,0.419355,4.901025


In [ ]:
# Initialize MinMaxScaler
scaler = MinMaxScaler()

# Identify numerical columns for scaling
numeric_cols = demo_X.select_dtypes(include=np.number).columns

# Apply scaler to numerical columns
demo_X[numeric_cols] = scaler.fit_transform(demo_X[numeric_cols])

print("Data after normalization (first 5 rows):")
demo_X.head()

Data after normalization (first 5 rows):


,age,gender,study_hours_per_day,academic_performance,exam_pressure,stress_level,sleep_hours,physical_activity,financial_stress,family_expectation,burnout_score,free_hours,study_sleep_ratio,external_pressure
987231,0.166667,1,0.228571,0.512699,0.404230,0.334131,0.428571,0.427963,0.499830,0.604895,0.061596,0.685279,0.131148,0.481685
79954,0.333333,0,0.564286,0.562701,0.640013,0.534745,0.457143,0.310108,0.182956,0.737223,0.261870,0.436548,0.313326,0.491926
567130,0.000000,0,0.228571,0.490037,0.521271,0.356646,0.585714,0.427963,0.694542,0.554551,0.000000,0.629442,0.110829,0.574140
500891,0.000000,1,0.400000,0.473927,0.555679,0.424690,0.014286,0.427963,0.727845,0.257658,0.460867,0.710660,0.444209,0.488010
55399,0.500000,1,0.185714,0.449923,0.336762,0.307238,0.457143,0.408459,0.647302,0.419919,0.005782,0.705584,0.103120,0.445317
